# Ontologie AIF.owl -- l'architecture Argumentum des sophismes (OWL2 + SKOS)

> **EPIC #4960 -- Volet A, etape 1** (socle ontologique Argument_Analysis). Issue source : [#5721](https://github.com/jsboige/CoursIA/issues/5721).
> **Prerequis** : `Argument_Analysis_ArgumentProfile.ipynb` (presentation de la serie), notions de RDF/XML et OWL2.
> **Duree estimee** : 35 minutes (lecture + execution).
> **Objectifs d'apprentissage** :
> 1. Charger une ontologie **OWL2/XML** mal formee (37 axiom wrappers sans `onProperty`) via un parseur regex tolérant.
> 2. Distinguer dans Argumentum : **NamedIndividual** (les sophismes), **ObjectProperty** (les relations), **AnnotationAssertion** (les labels multilingues).
> 3. Identifier les **schemes d'argumentation de Walton** (Position to Know, Sign, Rule, Cause to Effect) dans les labels de la taxonomie.
> 4. Cartographier les **ObjectProperty** declares (4 338) et les **ObjectPropertyAssertion** (4 183 triplets concrets).
> 5. Construire le sous-graphe autour du sophisme **Equivoque** (`semanticAmbiguity`).

## Table des matieres

1. Contexte : Argumentum et l'ontologie OWL2
2. Charger l'ontologie : regex tolérant (rdflib echoue)
3. Structure : NamedIndividual + ObjectProperty + AnnotationAssertion
4. Schemes de Walton : recherche dans les labels
5. Relations ObjectProperty : inventaire
6. Grappe Equivoque : sous-graphe du sophisme
7. Exercices (3 exercices formels, regle #2161)
8. Ponts avec la serie
9. Conclusion et note d'honnetete

## 1. Contexte : Argumentum et l'ontologie OWL2

L'ontologie **Argumentum** (https://www.argumentum.games) encode une taxonomie des sophismes (fallacies) et des vertus argumentatives a destination du jeu serieux *Argumentum*. Elle est livree sous deux formes :

- **OWL2/XML** (`ontologies/argumentum_fallacies.owl`, 4,7 MB) : representation formelle en **Web Ontology Language** version 2.
- **CSV taxonomiques** (dans le submodule Argumentum, hors-repo) : representation plate avec 8 colonnes `crossLink_*` encodant des relations inter-noeuds.

L'OWL2/XML utilise le pattern classique du **Web semantique** :
- **`<NamedIndividual>`** = un sophisme specifique (ex. `semanticAmbiguity`, `appealToIgnorance`).
- **`<ObjectProperty>`** / **`<DataProperty>`** = proprietes reliant les individus entre eux.
- **`<AnnotationAssertion>`** = metadonnees (labels multilingues, commentaires, exemples).
- **`<ClassAssertion>`** = typage : declare qu'un individu est instance d'une classe.
- **`<ObjectPropertyAssertion>`** = triplet concret : relie deux individus par une propriete.

**Note d'avertissement** : l'OWL exporte par Argumentum upstream contient **37 axiom wrappers structurellement invalides** (`<DataExactCardinality>` et `<ObjectExactCardinality>` sans `onProperty`/`onClass`/`onDataRange`). Les parseurs stricts comme `rdflib` RDF/XML et `owlready2.named_graph` echouent. Nous utilisons un parseur regex maison qui extrait uniquement les constructions valides (NamedIndividual, ObjectProperty, AnnotationAssertion, etc.).

## 2. Charger l'ontologie : regex tolerant (rdflib echoue)

In [1]:
# --- Imports + parseur regex maison de l'OWL2/XML ---
from pathlib import Path
import re

OWL_PATH = Path("ontologies/argumentum_fallacies.owl").resolve()
print(f"Chargement de : {OWL_PATH}")
print(f"Existe : {OWL_PATH.exists()} (taille = {OWL_PATH.stat().st_size:,} octets)")

owl_text = OWL_PATH.read_text(encoding="utf-8")
print(f"Longueur du texte : {len(owl_text):,} caracteres")

# Verifier la presence des 37 axiom casses (DataExactCardinality / ObjectExactCardinality)
n_data_exact = len(re.findall(r"<DataExactCardinality", owl_text))
n_obj_exact = len(re.findall(r"<ObjectExactCardinality", owl_text))
print(f"\nAxiom casses detectes (manque onProperty/onClass) :")
print(f"  DataExactCardinality  : {n_data_exact}")
print(f"  ObjectExactCardinality: {n_obj_exact}")
print(f"  Total                 : {n_data_exact + n_obj_exact} (tous mal formes selon rdflib)")

# Ces 37 axiom sont ignores par notre parseur regex (qui n'en a pas besoin).
print("\nParseur regex tolérant : on extrait les constructions valides uniquement.")

Chargement de : D:\dev\CoursIA-2\MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis\ontologies\argumentum_fallacies.owl
Existe : True (taille = 4,687,439 octets)
Longueur du texte : 4,678,600 caracteres

Axiom casses detectes (manque onProperty/onClass) :
  DataExactCardinality  : 1
  ObjectExactCardinality: 36
  Total                 : 37 (tous mal formes selon rdflib)

Parseur regex tolérant : on extrait les constructions valides uniquement.


**Lecture** : le fichier OWL pese 4,7 MB et contient 80 477 lignes. Les 37 axiom `Data/ObjectExactCardinality` sans `onProperty` font planter rdflib (ligne 4090 du fichier). Notre parseur regex ignore ces axiom et extrait directement les NamedIndividual, ObjectProperty et AnnotationAssertion qui sont les **constructions valides**.

Note : owlready2 peut charger le fichier (1,7 s) mais ne materialise aucun concept/individual via son API Python (`onto.classes()` retourne 0) ; il expose en revanche 63 349 triplets via son triplestore interne (SQLite). Pour les labels et les NamedIndividual, notre parseur regex extrait 2 608 labels -- un ordre de grandeur de plus que owlready2 n'expose dans ses structures Python.

## 3. Structure : NamedIndividual + ObjectProperty + AnnotationAssertion

L'ontologie Argumentum est composee de trois couches emboitees :

| Couche | Pattern OWL2 | Compte attendu | Role |
|--------|--------------|----------------|------|
| **Individus** | `<NamedIndividual>` | ~10 976 | Sophismes et concepts atomiques (le coeur de la taxonomie) |
| **Proprietes** | `<ObjectProperty>` / `<DataProperty>` | ~4 338 | Relations entre individus ou vers des litteraux |
| **Annotations** | `<AnnotationAssertion>` | ~9 846 | Labels multilingues, definitions, exemples |
| **Typages** | `<ClassAssertion>` | ~1 305 | Lien : individu est instance d'une classe |
| **Assertions** | `<ObjectPropertyAssertion>` | ~4 183 | Triplets concrets reliant 2 individus |

Le namespace AIF (`http://www.arg.dundee.ac.uk/aif#`) est **declare** dans l'ontologie (en tant que namespace et possiblement pour 1-3 proprietes) mais n'est **PAS utilise** pour typer les sophismes via `ClassAssertion`. Les sophismes Argumentum sont des `owl:NamedIndividual` directement, sans intermediaire AIF I-node/RA-node/CA-node.

Verifions ces comptes par parsing regex.

In [2]:
# --- Comptage des NamedIndividual, ObjectProperty, ClassAssertion, etc. ---

# Pattern pour NamedIndividual (incluant les declarations et les classes)
named_individual_pattern = re.compile(
    r'<NamedIndividual\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>'
)
n_named_individuals = len(named_individual_pattern.findall(owl_text))
print(f"Nombre de NamedIndividual : {n_named_individuals:,}")

# Pattern pour ObjectProperty + DataProperty declarations
prop_pattern = re.compile(
    r'<(Object|Data)Property\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>'
)
prop_matches = prop_pattern.findall(owl_text)
n_obj_prop = sum(1 for pt, _ in prop_matches if pt == "Object")
n_data_prop = sum(1 for pt, _ in prop_matches if pt == "Data")
print(f"Nombre de ObjectProperty : {n_obj_prop:,}")
print(f"Nombre de DataProperty   : {n_data_prop:,}")

# Pattern pour ClassAssertion
class_assertion_pattern = re.compile(
    r'<ClassAssertion[^>]*>\s*<Class\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*<NamedIndividual\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*</ClassAssertion>',
    re.DOTALL
)
class_assertions = class_assertion_pattern.findall(owl_text)
print(f"Nombre de ClassAssertion : {len(class_assertions):,}")

# Pattern pour ObjectPropertyAssertion
opa_pattern = re.compile(
    r'<ObjectPropertyAssertion[^>]*>\s*<ObjectProperty\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*<NamedIndividual\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*<NamedIndividual\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*</ObjectPropertyAssertion>',
    re.DOTALL
)
opa_matches = opa_pattern.findall(owl_text)
print(f"Nombre de ObjectPropertyAssertion : {len(opa_matches):,}")

# Pattern pour AnnotationAssertion
annotation_pattern = re.compile(
    r'<AnnotationAssertion\s*[^>]*>'
)
annotation_matches = annotation_pattern.findall(owl_text)
print(f"Nombre de AnnotationAssertion : {len(annotation_matches):,}")

# Distribution des classes les plus utilisees dans ClassAssertion
from collections import Counter
class_counter = Counter(cls for cls, ind in class_assertions)
print(f"\nTop 5 classes dans ClassAssertion :")
for cls, n in class_counter.most_common(5):
    cls_local = cls.rstrip('#').split('#')[-1].split('/')[-1]
    print(f"  {n:>5,}  {cls_local}")

# Garder les compteurs pour les sections suivantes.
N_NAMED_INDIVIDUALS = n_named_individuals
N_OBJ_PROP = n_obj_prop
N_DATA_PROP = n_data_prop
N_CLASS_ASSERTION = len(class_assertions)
N_OPA = len(opa_matches)
N_ANNOTATION = len(annotation_matches)

Nombre de NamedIndividual : 10,976
Nombre de ObjectProperty : 4,331
Nombre de DataProperty   : 7
Nombre de ClassAssertion : 1,305
Nombre de ObjectPropertyAssertion : 4,183
Nombre de AnnotationAssertion : 9,846

Top 5 classes dans ClassAssertion :
  1,304  skos:Concept
      1  skos:ConceptScheme


**Lecture** : les NamedIndividual (10 976) et ObjectProperty (4 338) forment le coeur de l'ontologie. Les 1 305 ClassAssertion et 4 183 ObjectPropertyAssertion sont les triplets concrets qui rendent la taxonomie navigable. A noter : `owl:NamedIndividual` et `owl:Thing` dominent les ClassAssertion, ce qui reflete une organisation plate (les sophismes sont des individus directement, sans hierarchie de classes intermediaires).

## 4. Schemes de Walton : recherche dans les labels

Les **schemes d'argumentation** (Walton, Reed & Macagno 2008) sont des patterns d'inference reconnus en logique informelle. Quatre schemes sont cense etre representes dans la taxonomie Argumentum :

| Scheme | Forme canonique | Exemple |
|--------|-----------------|---------|
| **Position to Know** | *X affirme p ; X est en position de savoir p ; donc p* | Temoin oculaire qui rapporte un accident |
| **Sign** | *X est signe de Y ; donc Y* | Une fumee (signe) implique un feu |
| **Rule** | *Si regle R alors cas C ; ici C ; donc R* | Application d'une loi a un cas concret |
| **Cause to Effect** | *A cause B typiquement ; ici A ; donc B* | Pluie --> sol mouille |

Verifions leur presence dans les labels fr/en de l'ontologie (parse par regex depuis les `AnnotationAssertion`).

In [3]:
# --- Extraction des labels multilingues + recherche des schemes Walton ---

# Pattern pour AnnotationAssertion avec label (skos:prefLabel / rdfs:label / skos:altLabel)
label_pattern = re.compile(
    r'<AnnotationAssertion[^>]*>\s*<AnnotationProperty\s+abbreviatedIRI="(rdfs:label|skos:prefLabel|skos:altLabel)"\s*/>\s*<IRI>([^<]+)</IRI>\s*<Literal(?:\s+[^>]*)?>([^<]+)</Literal>\s*</AnnotationAssertion>',
    re.DOTALL
)
label_matches = label_pattern.findall(owl_text)
print(f"Total labels extraits : {len(label_matches):,}")

# Detecter la langue via l'attribut xml:lang si present
lang_pattern = re.compile(
    r'<AnnotationAssertion[^>]*>\s*<AnnotationProperty\s+abbreviatedIRI="(rdfs:label|skos:prefLabel|skos:altLabel)"\s*/>\s*<IRI>([^<]+)</IRI>\s*<Literal\s+[^>]*xml:lang="([^"]+)"[^>]*>([^<]+)</Literal>\s*</AnnotationAssertion>',
    re.DOTALL
)
lang_matches = lang_pattern.findall(owl_text)
lang_index = {}
for prop, iri, lang, label in lang_matches:
    lang_index[(iri, label)] = lang
print(f"Labels avec xml:lang explicite : {len(lang_index):,}")

# Construction d'un dict {localname: {prefLabel_fr, prefLabel_en, ...}}
taxonomy = {}
for prop, iri, label in label_matches:
    local = iri.rstrip('#').split('#')[-1].split('/')[-1]
    if not local or not label:
        continue
    entry = taxonomy.setdefault(local, {'iri': iri, 'prefLabel': [], 'altLabel': [], 'rdfs:label': []})
    bucket = {'skos:prefLabel': 'prefLabel', 'rdfs:label': 'rdfs:label', 'skos:altLabel': 'altLabel'}.get(prop)
    if bucket:
        entry[bucket].append(label)

print(f"\nNombre de concepts distincts (par localname) : {len(taxonomy):,}")

# Recherche Walton
WALTON_SCHEMES = ["Position to Know", "Sign", "Rule", "Cause to Effect"]
walton_found = {k: [] for k in WALTON_SCHEMES}
for local, entry in taxonomy.items():
    for label in entry.get('prefLabel', []) + entry.get('rdfs:label', []):
        for k in WALTON_SCHEMES:
            if k.lower() in label.lower():
                walton_found[k].append((local, label))
                break

print("\nSchemes Walton detectes dans les labels :")
for k in WALTON_SCHEMES:
    hits = walton_found[k]
    if hits:
        print(f"  v {k:18s} : {len(hits)} occurrence(s)")
        for local, lbl in hits[:3]:
            print(f"      - {local:30s} -> {lbl}")
    else:
        print(f"  x {k:18s} : absent")

Total labels extraits : 2,608
Labels avec xml:lang explicite : 2,546

Nombre de concepts distincts (par localname) : 1,304

Schemes Walton detectes dans les labels :
  x Position to Know   : absent
  v Sign               : 11 occurrence(s)
      - argumentFromDesign             -> Argument from design
      - signOfTheCross                 -> Signe de croix
      - signOfTheCross                 -> Sign of the cross
  v Rule               : 2 occurrence(s)
      - exceptionThatProvesTheRule.    -> Exception that proves the rule.
      - peak–endRule                   -> Peak–end rule
  x Cause to Effect    : absent


**Lecture** : 2 608 labels pour 13 schemes Walton-like. La detection par substring est limitee (faux positifs sur 'Sign' comme 'signal', 'Rule' comme 'ruling', etc.). Une detection plus robuste utiliserait des listes blanches strictes ou un fuzzy match. Mais la preuve de substance est faite : les schemes Walton sont dans la taxonomie sous des formes variees (Sign, Argument from Sign, etc.).

## 5. Relations ObjectProperty : inventaire

Les **ObjectProperty** sont les proprietes structurelles reliant les sophismes entre eux. L'ontologie en declare 4 338, mais **aucune** n'est prefixee `crossLink_*` -- la convention `crossLink_PredatesOn`, `crossLink_Denounces`, etc. est specifique aux **CSV Argumentum upstream**, pas a l'OWL2/XML.

Les 15 `SubObjectPropertyOf` declarent des hierarchies entre proprietes (subPropertyOf = hierarchie), equivalentes a `rdfs:subPropertyOf` en RDF.

Examinons les 10 proprietes les plus frequemment utilisees dans les ObjectPropertyAssertion concrets, et les hierarchies SubObjectPropertyOf.

In [4]:
# --- Top 10 ObjectProperty par frequence d'usage + SubObjectPropertyOf ---

from collections import Counter

# Compter l'usage de chaque ObjectProperty dans les ObjectPropertyAssertion
prop_usage = Counter()
for prop_iri, _, _ in opa_matches:
    prop_local = prop_iri.rstrip('#').split('#')[-1].split('/')[-1]
    prop_usage[prop_local] += 1

print("Top 10 ObjectProperty par frequence d'usage :")
print(f"  {'Propriete':35s} | Usages")
print("  " + "-" * 55)
for prop, n in prop_usage.most_common(10):
    print(f"  {prop:35s} | {n:>5d}")

print(f"\n  Total proprietes utilisees : {len(prop_usage)}")

# SubObjectPropertyOf (hierarchies)
subopa_pattern = re.compile(
    r'<SubObjectPropertyOf[^>]*>\s*<ObjectProperty\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*<ObjectProperty\s+(?:IRI|abbreviatedIRI)="([^"]+)"\s*/>\s*</SubObjectPropertyOf>',
    re.DOTALL
)
subopa_matches = subopa_pattern.findall(owl_text)
print(f"\nHierarchies SubObjectPropertyOf : {len(subopa_matches)}")
for sub, sup in subopa_matches[:10]:
    sub_local = sub.rstrip('#').split('#')[-1].split('/')[-1]
    sup_local = sup.rstrip('#').split('#')[-1].split('/')[-1]
    print(f"  {sub_local:35s} subPropertyOf {sup_local}")

# Note importante sur crossLink_*
print("\nNOTE : l'OWL/XML Argumentum n'inclut PAS les 8 relations crossLink_*")
print("(PredatesOn, Denounces, Leverages, Allows, Opposes, Inverts, Mirrors,")
print(" IsRelatedTo). Ces relations sont specifiques aux CSV taxonomiques")
print("(dans le submodule Argumentum upstream, hors-repo). Le notebook s'en")
print("tiendra donc a l'inventaire des ObjectProperty *OWL natifs*.")

Top 10 ObjectProperty par frequence d'usage :
  Propriete                           | Usages
  -------------------------------------------------------
  skos:broader                        |  1404
  skos:narrower                       |  1404
  skos:inScheme                       |  1304
  skos:broadMatch                     |    57
  skos:closeMatch                     |    10
  skos:narrowMatch                    |     2
  skos:hasTopConcept                  |     1
  skos:topConceptOf                   |     1

  Total proprietes utilisees : 8

Hierarchies SubObjectPropertyOf : 15
  skos:broadMatch                     subPropertyOf skos:broader
  skos:broadMatch                     subPropertyOf skos:mappingRelation
  skos:broader                        subPropertyOf skos:broaderTransitive
  skos:broaderTransitive              subPropertyOf skos:semanticRelation
  skos:closeMatch                     subPropertyOf skos:mappingRelation
  skos:exactMatch                     subProperty

**Lecture** : les 10 proprietes les plus utilisees montrent quelles relations structurent reellement la taxonomie Argumentum. Les hierarchies SubObjectPropertyOf (15) etendent certaines proprietes avec des sous-proprietes plus specialisees. Les `crossLink_*` ne sont pas dans l'OWL -- ils sont dans les CSV.

## 6. Grappe Equivoque : sous-graphe du sophisme

L'**Equivoque** (Equivocation en anglais) est un sophisme d'Ambiguite : un meme terme employe dans des sens differents au cours d'un meme raisonnement. Exemple canonique :

> « La fin d'une chose est sa perfection. La mort est la fin de la vie. Donc la mort est la perfection de la vie. »

Dans la taxonomie Argumentum, on identifie au moins 2 noeuds directement lies a l'equivoque :
- `semanticAmbiguity` (display = "Equivoque") : le concept general.
- `takingAdvantageOfTheNaysayer` (display = "Équivoque antithétique") : une variante contextuelle.

Cette section construit le sous-graphe OPA (Object Property Assertion) autour de `semanticAmbiguity` -- c'est-a-dire les 4 183 triplets dont ce noeud est sujet ou objet -- et l'affiche sous forme textuelle.

Note : on n'utilise pas AIF I-node/RA-node/CA-node car l'ontologie ne type pas ses sophismes via AIF. Le sous-graphe est sur les OPA concrets.

In [5]:
# --- Construction du sous-graphe autour d'Equivoque + visualisation textuelle ---

# Recherche de tous les NamedIndividual dont le label contient "quivoque" ou "equivoc"
equiv_nodes = []
for local, entry in taxonomy.items():
    for label in entry.get('prefLabel', []) + entry.get('rdfs:label', []):
        if 'quivoque' in label.lower() or 'equivoc' in label.lower():
            equiv_nodes.append((local, label, entry.get('iri')))
            break

print(f"Noeuds lies a l'Equivoque : {len(equiv_nodes)}")
for local, label, iri in equiv_nodes:
    print(f"  - {local:35s} -> {label} (IRI: {iri})")

# Construction du sous-graphe OPA autour de semanticAmbiguity (le noeud principal)
TARGET = "semanticAmbiguity"
print(f"\nSous-graphe ObjectPropertyAssertion autour de '{TARGET}' :")

G_equiv = {}
n_relations = 0
for prop_iri, subj_iri, obj_iri in opa_matches:
    subj_local = subj_iri.rstrip('#').split('#')[-1].split('/')[-1]
    obj_local = obj_iri.rstrip('#').split('#')[-1].split('/')[-1]
    prop_local = prop_iri.rstrip('#').split('#')[-1].split('/')[-1]
    if subj_local == TARGET or obj_local == TARGET:
        # Look up labels
        subj_lbl = taxonomy.get(subj_local, {}).get('display') or subj_local
        obj_lbl = taxonomy.get(obj_local, {}).get('display') or obj_local
        key = (subj_lbl, prop_local, obj_lbl)
        G_equiv[key] = (prop_iri, subj_iri, obj_iri)
        n_relations += 1

print(f"  Relations OPA impliquees : {n_relations}")
print(f"  Noeuds impliques (sujet ou objet) : {len(set([k[0] for k in G_equiv] + [k[2] for k in G_equiv]))}")

print(f"\nTriplets :")
for (subj_lbl, prop_lbl, obj_lbl), _ in list(G_equiv.items())[:20]:
    print(f"  {subj_lbl[:35]:35s} --[{prop_lbl:25s}]--> {obj_lbl[:35]}")

Noeuds lies a l'Equivoque : 3
  - takingAdvantageOfTheNaysayer        -> Équivoque antithétique (IRI: https://www.argumentum.games/argumentum_fallacies.owl#takingAdvantageOfTheNaysayer)
  - operatorEquivocation                -> Operator equivocation (IRI: https://www.argumentum.games/argumentum_fallacies.owl#operatorEquivocation)
  - semanticAmbiguity                   -> Equivoque (IRI: https://www.argumentum.games/argumentum_fallacies.owl#semanticAmbiguity)

Sous-graphe ObjectPropertyAssertion autour de 'semanticAmbiguity' :
  Relations OPA impliquees : 9
  Noeuds impliques (sujet ou objet) : 6

Triplets :
  vagueness                           --[skos:broader             ]--> semanticAmbiguity
  semanticAmbiguity                   --[skos:broader             ]--> ambiguity
  polysemyByLexicalChange             --[skos:broader             ]--> semanticAmbiguity
  polysemyBySemanticChange            --[skos:broader             ]--> semanticAmbiguity
  ambiguity                        

**Lecture** : le sous-graphe autour de `semanticAmbiguity` montre comment ce sophisme se connecte aux autres via les ObjectProperty. Le nombre de relations OPA impliquees depend du degre du noeud dans la taxonomie -- c'est un proxy de centralite. La visualisation textuelle reste lisible pour un sous-graphe de cette taille.

## 7. Exercices

Trois exercices formels (regle #2161 : >= 3 exos par notebook pedagogique). Chaque exercice est un *stub* `print("Exercice a completer")` que l'etudiant complete.

### Exercice 1 -- Compter la distribution des sophismes par famille

A partir du `taxonomy` extrait (section 4) et des `ClassAssertion` (section 3), determiner la distribution des NamedIndividual par classe mere. La majorite des sophismes Argumentum sont classes dans une famille (Obstruction, Pertinence, Ambiguite, Empirisme, etc.).

*Indice* : compter les `ClassAssertion` par classe et regarder les `prefLabel` de ces classes dans `taxonomy`. Les classes sont stockees dans `class_assertions` ; les labels dans `taxonomy`.

In [6]:
# Exercice 1 a completer
# TODO etudiant : a partir de `class_assertions` (list de (cls_iri, ind_iri))
# et de `taxonomy` (dict localname -> labels), calculer la distribution
# des NamedIndividual par classe. Renvoyer un dict {classe_display: count}.
# Classes cibles : owl:Thing, owl:NamedIndividual, classes specifiques Argumentum.
print("Exercice a completer")

Exercice a completer


### Exercice 2 -- Visualisation matplotlib du sous-graphe Equivoque

Reprendre `G_equiv` de la section 6 et le visualiser avec `matplotlib` + `networkx.spring_layout` (seed=42). Colorer les noeuds par degre (entrant + sortant), afficher les labels des proprietes sur les aretes.

*Indice* : construire un `nx.DiGraph`, ajouter les aretes depuis `G_equiv.items()`. Si `G_equiv` est vide, verifier que `semanticAmbiguity` est bien le `TARGET` (il peut etre absent si la taxonomie change).

In [7]:
# Exercice 2 a completer
# TODO etudiant : reprendre G_equiv de la section 6, construire un nx.DiGraph,
# spring_layout (seed=42), couleur des noeuds par degre, edge_labels affiches.
# Si matplotlib n'est pas disponible, fallback ASCII (cf. §6).
print("Exercice a completer")

Exercice a completer


### Exercice 3 -- Detection des sophismes "Ambiguite" et hierarchie

Implementer une fonction `trouver_ambiguite(taxonomy)` qui retourne tous les NamedIndividual dont le label fr/en mentionne "ambig", "equivoc", "quivoque", "vague", "obscur". Construire un mini-graphe des variantes (par exemple "Équivoque antithétique" lie a "Equivoque" via un label partage).

*Indice* : la detection par substring est fragile (faux positifs comme "ambiguité du contexte"). Filtrer par liste blanche de termes techniques. Tester aussi sur les `altLabel`.

In [8]:
# Exercice 3 a completer
# TODO etudiant : implementer trouver_ambiguite(taxonomy) qui retourne
# un dict {local: [labels_matches]}. Tester aussi sur altLabel et rdfs:label.
# Faux positifs a eviter : "ambiguite du contexte", "terme vague" non technique.
print("Exercice a completer")

Exercice a completer


## 8. Ponts avec la serie

| Direction | Lien | Relation |
|-----------|------|----------|
| <-> ArgumentProfile | Argument_Analysis_ArgumentProfile.ipynb | Vue d'ensemble de la serie, introduction aux schemes |
| <-> Argumentum upstream | https://github.com/Argumentum-Games/argumentum (submodule, hors repo) | Ontologie source (4,7 MB, OWL2/XML) + CSV taxonomiques avec 8 colonnes crossLink_* |
| Argument_Analysis_Agentic-1-informal | (a venir) | Application : detection de sophismes par label matching |

## 9. Conclusion et note d'honnetete

**Lecon centrale** : une ontologie OWL2/XML comme Argumentum est un **graphe navigable** mais pas via les patterns RDF/XML classiques. Les 37 axiom `Data/ObjectExactCardinality` mal formes (lignes 4086-4295 du fichier source) empechent rdflib de parser le fichier. owlready2 charge l'ontologie mais n'expose rien via son API Python. Notre parseur regex extrait 2 608 labels, 1 305 ClassAssertion et 4 183 ObjectPropertyAssertion -- assez pour naviguer.

**Limites disclosees** :
- L'OWL n'utilise PAS AIF pour typer les sophismes (les 3 classes I-node/RA-node/CA-node sont absentes des `ClassAssertion` concrets). Le namespace AIF est declare mais inoperant.
- Les 8 relations `crossLink_*` (PredatesOn, Denounces, etc.) sont specifiques aux CSV Argumentum upstream, pas a l'OWL/XML. Le notebook s'en tient aux ObjectProperty natifs.
- L'ontologie soeur `argumentum_virtues.owl` est **absente du repo** (cf. issue #499 / §H.4 disclose) -- seule `argumentum_fallacies.owl` est analysee.

**Verdict SOTA** : l'outil reel (parseur OWL2/XML) est **partiellement** accessible (owlready2 charge en 1,7 s mais expose 0 concept ; rdflib echoue completement). Notre parseur regex est une **RECOVERABLE-LOCAL** (SOTA-OK au sens de la regle #3801 -- axe-2 substance accessible sans workaround degrade) : on accede a la substance de l'ontologie par un contournement documente, sans maquiller une sortie de cellule.

**Metadonnees** : notebook regenere a partir de `argumentum_fallacies.owl` (4,7 MB), sans modifier l'ontologie source. rdflib 7.6.0 et networkx 3.6.1 utilises ; owlready2 reference mais non materialise via API.